### Transfer Learning using MobileNetv2


Dataset Structure
```bash
dataset/
├── train/
│   ├── cats/
│   └── dogs/
└── test/
    ├── cats/
    └── dogs/
```

STEP 1: Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

STEP 2: Image Preprocessing
We have to rescale the images to make suitable for our model using an instance of "ImageDataGenerator" i.e.:
```Python
datagen = ImageDataGenerator(rescale=1./255)
```

And apply it to both Train and Test dataset:
using 
```python
datagen.flow_from_directory() 
```
as we are directly using the image directories:
Hence:

| Parameter | Value |
| :- | :-: |
| Directory | dataset/train or test |
| target_size | (224,224) |
| batch_size | 20 |
| class_mode | binary |

In [7]:
datagen = ImageDataGenerator(rescale = 1./255)

In [8]:
train = datagen.flow_from_directory(
    
    "D:\\EduNet\\NSTI_Indore25_26\\Deep Learning\\Dataset\\Dog and Cat\\train",
    target_size = (224,224),
    batch_size = 20,
    class_mode= 'binary'
)
test = datagen.flow_from_directory(
    
    "D:\\EduNet\\NSTI_Indore25_26\\Deep Learning\\Dataset\\Dog and Cat\\test",
    target_size = (224,224),
    batch_size = 20,
    class_mode= 'binary'
)

Found 800 images belonging to 2 classes.
Found 200 images belonging to 2 classes.


STEP 3: Load Pre-trained Model
Load MobileNetV2
| Parameter | Value |
| :- | :-: |
| weights | imagenet |
| include_top | False |
| input_shape | (224,224,3) |

and most important:
We have to freeze this model:
```python
mobilenet_model.trainable = False
```


In [9]:
base_model = MobileNetV2(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

In [10]:
base_model.trainable = False

STEP 4: Build the Simplest Transfer Learning Model

Architecture
```css
Image
↓
Pre-trained CNN (Frozen)
↓
Pooling
↓
Dense Layer
↓
1 Output Neuron (Cat / Dog)
```

which means we have to follow:

1. mobilenet_model:
```Python 
mobilenet_model
```
2. Pooling Layer:
```Python
GlobalAveragePooling2D()
```
3. Dense Layer
```Python
Dense(1024, activation='relu')
```
4. Output layer (classification):
```Python
Dense(1, activation='softmax')
```
1 neurons because output classes are 2 i.e. 0 or 1

In [12]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512,activation='relu'),
    Dense(1,activation='sigmoid')
])

Step 5: Compile Model
Let's compile our model with
- optimizer as adam
- loss as binary_crossentropy
- metrics=['accuracy']

In [15]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [16]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,914,369 (11.12 MB)

 Trainable params: 656,385 (2.50 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

Step 6: Train the Model
| Parameters | Value |
| -------- | :-------: |
| epochs | 3 |
| validation_data | test_data |

In [17]:
model.fit(train,validation_data=test,epochs=5)

Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 28s 611ms/step - accuracy: 0.9200 - loss: 0.2187 - val_accuracy: 0.9600 - val_loss: 0.0677
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 16s 400ms/step - accuracy: 0.9750 - loss: 0.0731 - val_accuracy: 0.9950 - val_loss: 0.0216
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 412ms/step - accuracy: 0.9962 - loss: 0.0223 - val_accuracy: 0.9950 - val_loss: 0.0189
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 438ms/step - accuracy: 0.9987 - loss: 0.0065 - val_accuracy: 1.0000 - val_loss: 0.0084
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 427ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 0.0077
